# Stage 8 — XGBoost trên CICIoMT2024

Notebook fail-closed cho đúng một experiment/config/run. Mặc định dry-run; không tự dùng Test, không ghi đè run cũ và không tự resume.

In [ ]:
# CELL 1 — Runtime inventory (read-only)
import json, os, platform, shutil, subprocess, sys
from pathlib import Path
import psutil
print({
    'python': sys.version, 'platform': platform.platform(),
    'logical_cpus': os.cpu_count(), 'physical_cpus': psutil.cpu_count(logical=False),
    'ram_total_gib': psutil.virtual_memory().total / 2**30,
    'ram_available_gib': psutil.virtual_memory().available / 2**30,
    'disk_free_gib': shutil.disk_usage('/content').free / 2**30,
})
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU inventory only (not auto-selected):', gpu.stdout.strip() if gpu.returncode == 0 else 'NONE')

In [ ]:
# CELL 2 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
DRIVE_ROOT = Path('/content/drive/MyDrive/COMPACT-MI-RFE')
if not DRIVE_ROOT.is_dir():
    raise FileNotFoundError(f'MISSING Drive project root: {DRIVE_ROOT}')
print('Drive root:', DRIVE_ROOT)

In [ ]:
# CELL 3 — Clone/select an exact clean Git commit
REPO_URL = 'https://github.com/kasum00/RFforNetwork.git'
APPROVED_COMMIT = 'REPLACE_WITH_40_HEX_COMMIT'  #@param {type:'string'}
REPO = Path('/content/RFforNetwork')
if len(APPROVED_COMMIT) != 40 or any(c not in '0123456789abcdefABCDEF' for c in APPROVED_COMMIT):
    raise ValueError('Set APPROVED_COMMIT to the exact reviewed 40-hex commit')
if not REPO.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'switch', '--detach', APPROVED_COMMIT], check=True)
actual = subprocess.run(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(['git', '-C', str(REPO), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout.strip()
if actual.lower() != APPROVED_COMMIT.lower() or dirty:
    raise RuntimeError({'actual_commit': actual, 'dirty': dirty})
print('Clean detached checkout:', actual)

In [ ]:
# CELL 4 — Source hashes; no fit and no dataset rows
import hashlib
def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest().upper()
STAGE8_SOURCE = REPO / 'huyen_code' / 'ciciomt24_xgboost.py'
TRAINING_CORE = REPO / 'huyen_code' / 'ciciomt24_training.py'
for required in (STAGE8_SOURCE, TRAINING_CORE, REPO / 'huyen_code' / 'test_ciciomt24_xgboost.py'):
    if not required.is_file(): raise FileNotFoundError(required)
SOURCE_HASHES = {'stage8_sha256': file_sha256(STAGE8_SOURCE), 'training_core_sha256': file_sha256(TRAINING_CORE)}
print(json.dumps(SOURCE_HASHES, indent=2))

In [ ]:
# CELL 5 — Reuse existing WINDOWS_CPLUS; never download/regenerate
WINDOW_ROOT = DRIVE_ROOT / 'CICIoMT24' / 'WINDOWS_CPLUS'  #@param {type:'string'}
WINDOW_ROOT = Path(WINDOW_ROOT)
required = WINDOW_ROOT / 'window_run_summary.json'
if not WINDOW_ROOT.is_dir() or not required.is_file():
    raise FileNotFoundError(f'MISSING prepared dataset: {WINDOW_ROOT}')
summary = json.loads(required.read_text(encoding='utf-8'))
if summary.get('status') != 'PASS' or summary.get('feature_columns') != 380:
    raise ValueError('WINDOWS_CPLUS summary is not the validated 380-feature dataset')
print({'dataset': str(WINDOW_ROOT), 'summary_sha256': file_sha256(required), 'verified_windows': summary.get('readback_validation', {}).get('verified_rows')})

In [ ]:
# CELL 6 — Install/check exact Stage 8 Colab profile
REQ = REPO / 'requirements-stage8-colab.txt'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(REQ)], check=True)
import importlib.metadata as metadata
expected = {'numpy':'2.2.6','pandas':'2.3.3','scipy':'1.15.3','scikit-learn':'1.7.2','joblib':'1.5.3','psutil':'5.9.5','xgboost':'3.4.1'}
actual = {name: metadata.version(name) for name in expected}
loaded_psutil = psutil.__version__
if actual != expected or loaded_psutil != expected['psutil']:
    raise RuntimeError('Restart runtime after installation, then rerun from Cell 1. expected=%r installed=%r loaded_psutil=%r' % (expected, actual, loaded_psutil))
print('Stage 8 dependency profile PASS:', actual)

In [ ]:
# CELL 7 — Unit tests + read-only dataset validation
SESSION = Path('/content/stage8_session')
SESSION.mkdir(exist_ok=True)
env = dict(os.environ)
env['PYTHONPATH'] = str(REPO) + os.pathsep + str(REPO / 'huyen_code')
test = subprocess.run([sys.executable, '-m', 'unittest', '-v', 'huyen_code.test_ciciomt24_xgboost'], cwd=REPO, env=env, capture_output=True, text=True)
print(test.stdout); print(test.stderr)
UNIT_EVIDENCE = SESSION / 'unit_tests.json'
UNIT_EVIDENCE.write_text(json.dumps({
    'status': 'PASS' if test.returncode == 0 else 'FAIL', 'returncode': test.returncode,
    **SOURCE_HASHES, 'command': 'python -m unittest -v huyen_code.test_ciciomt24_xgboost'
}, indent=2), encoding='utf-8')
if test.returncode != 0: raise RuntimeError('Stage 8 tests FAIL; full training blocked')
sys.path.insert(0, str(REPO))
from colab.dataset_validator import validate_dataset
dataset_audit = validate_dataset(WINDOW_ROOT, REPO, all_headers=True)
print('Dataset validation PASS; this check reads headers/metadata, not Test feature rows')
print(json.dumps(dataset_audit, indent=2, default=str)[:12000])

In [ ]:
# CELL 8 — Select exactly one experiment; default is dry-run
EXPERIMENT = 'baseline_xgb'  #@param ['baseline_xgb', 'mi_rfe_xgb']
MODE = 'dry_run'  #@param ['dry_run', 'full_train']
BACKEND = 'cpu'  #@param ['cpu', 'cuda']
if EXPERIMENT not in ('baseline_xgb', 'mi_rfe_xgb'): raise ValueError(EXPERIMENT)
if MODE not in ('dry_run', 'full_train'): raise ValueError(MODE)
if BACKEND not in ('cpu', 'cuda'): raise ValueError(BACKEND)
print({'experiment': EXPERIMENT, 'mode': MODE, 'backend': BACKEND, 'cpu_reference': BACKEND == 'cpu'})

In [ ]:
# CELL 9 — Select one FW/FS and a unique run name
CONFIG = 'FW_512_FS_16'  #@param ['FW_256_FS_8','FW_256_FS_16','FW_512_FS_8','FW_512_FS_16']
RUN_NAME = 'stage8_baseline_xgb_FW512_FS16_run01'  #@param {type:'string'}
if not RUN_NAME or any(c not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for c in RUN_NAME):
    raise ValueError('RUN_NAME may contain only letters, digits, _ and -')
RESULT_ROOT = DRIVE_ROOT / 'experiment_results'
OUTPUT = RESULT_ROOT / RUN_NAME
CACHE_ROOT = Path('/content/stage8_cache') / RUN_NAME
RFE_POLICY = REPO / 'configs' / 'stage8_xgboost_policy.BLOCKED.json'
print({'config': CONFIG, 'run_name': RUN_NAME, 'output': str(OUTPUT), 'cache': str(CACHE_ROOT), 'rfe_policy': str(RFE_POLICY)})
print('MI-RFE+XGB stays BLOCKED until RFE_POLICY points to a separately approved policy JSON.')

In [ ]:
# CELL 10 — Dry preflight only; no model.fit and no Test feature rows
BUDGET = SESSION / f'{RUN_NAME}.budget.json'
budget = {
    'max_elapsed_seconds': 43200,
    'max_tree_rss_bytes': 40 * 2**30,
    'min_start_available_ram_bytes': 44 * 2**30,
    'min_runtime_available_ram_bytes': 4 * 2**30,
    'min_free_disk_bytes': 40 * 2**30,
    'poll_interval_seconds': 10,
}
BUDGET.write_text(json.dumps(budget, indent=2), encoding='utf-8')
PREFLIGHT = SESSION / f'{RUN_NAME}.preflight.json'
if PREFLIGHT.exists(): PREFLIGHT.unlink()
cmd = [sys.executable, str(STAGE8_SOURCE), 'preflight', '--repo', str(REPO), '--windows', str(WINDOW_ROOT), '--original', str(REPO/'CODE_ctu13'), '--output', str(OUTPUT), '--cache-root', str(CACHE_ROOT), '--config', CONFIG, '--experiment', EXPERIMENT, '--backend', BACKEND, '--budget', str(BUDGET), '--unit-test-evidence', str(UNIT_EVIDENCE), '--rfe-policy', str(RFE_POLICY), '--report', str(PREFLIGHT)]
preflight_run = subprocess.run(cmd, cwd=REPO, env=env, text=True, capture_output=True)
print(preflight_run.stdout); print(preflight_run.stderr)
if not PREFLIGHT.is_file():
    raise RuntimeError(f'Preflight did not produce a complete JSON report; returncode={preflight_run.returncode}')
try:
    audit = json.loads(PREFLIGHT.read_text(encoding='utf-8'))
except (FileNotFoundError, json.JSONDecodeError) as exc:
    raise RuntimeError('Preflight did not produce a complete strict-JSON report') from exc
print('PREFLIGHT STATUS:', audit['status']); print(json.dumps(audit['checks'], indent=2))
print('NO FIT / NO TEST:', audit['model_fit_calls'], audit['test_feature_rows_read'])

In [ ]:
# CELL 11 — Explicit resource approval and exact confirmation
import secrets
I_ACCEPT_RESOURCE_BUDGET = False  #@param {type:'boolean'}
CONFIRMATION = ''
SESSION_APPROVAL_TOKEN = None
EXPECTED_CONFIRMATION = f'RUN STAGE8 {EXPERIMENT} {CONFIG} {RUN_NAME} {BACKEND}'
print('Resource budget:', json.dumps(budget, indent=2))
print('Required exact confirmation:', EXPECTED_CONFIRMATION)
if MODE == 'full_train':
    if not I_ACCEPT_RESOURCE_BUDGET: raise PermissionError('Resource budget not accepted')
    CONFIRMATION = input('Type the exact confirmation now: ').strip()
    if CONFIRMATION != EXPECTED_CONFIRMATION: raise PermissionError('Exact confirmation mismatch')
    SESSION_APPROVAL_TOKEN = secrets.token_hex(16)
    print('Interactive session approval created; run Cell 12 separately.')
else:
    print('DRY RUN: no interactive approval created.')

In [ ]:
# CELL 12 — Launch exactly one supervised official run
if MODE == 'dry_run':
    print('DRY RUN ONLY — no training started')
else:
    if audit['status'] != 'PASS': raise RuntimeError('Preflight BLOCKED')
    if not I_ACCEPT_RESOURCE_BUDGET: raise PermissionError('Resource budget not accepted')
    if not SESSION_APPROVAL_TOKEN: raise PermissionError('Run Cell 11 interactively; Run all cannot create a launch token')
    if CONFIRMATION != EXPECTED_CONFIRMATION: raise PermissionError('Exact confirmation mismatch')
    supervisor_preflight = SESSION / f'{RUN_NAME}.supervisor_preflight.json'
    if supervisor_preflight.exists(): raise FileExistsError(supervisor_preflight)
    cmd = [sys.executable, str(STAGE8_SOURCE), 'supervise', '--repo', str(REPO), '--windows', str(WINDOW_ROOT), '--original', str(REPO/'CODE_ctu13'), '--output', str(OUTPUT), '--cache-root', str(CACHE_ROOT), '--config', CONFIG, '--experiment', EXPERIMENT, '--backend', BACKEND, '--budget', str(BUDGET), '--unit-test-evidence', str(UNIT_EVIDENCE), '--rfe-policy', str(RFE_POLICY), '--preflight-report', str(supervisor_preflight), '--confirmation', CONFIRMATION]
    completed = subprocess.run(cmd, cwd=REPO, env=env)
    if completed.returncode != 0: raise RuntimeError(f'Worker stopped/failed: {completed.returncode}')
    print('Supervisor completed; verify artifacts in Cells 13–14.')

In [ ]:
# CELL 13 — Verify saved result only; never predict/evaluate Test again
if not OUTPUT.is_dir():
    print('No completed result directory yet:', OUTPUT)
else:
    result = json.loads((OUTPUT/'result.json').read_text(encoding='utf-8'))
    validated = json.loads((OUTPUT/'RUN_VALIDATED.json').read_text(encoding='utf-8'))
    manifest = json.loads((OUTPUT/'artifact_manifest.json').read_text(encoding='utf-8'))
    required = ['model.ubj','final_scaler.joblib','features.json','hyperparameters.json','result.json','TEST_CLAIM.json','RUN_VALIDATED.json']
    missing = [name for name in required if not (OUTPUT/name).is_file()]
    if missing: raise FileNotFoundError(missing)
    if result['test_evaluation_calls'] != 1 or validated['test_evaluation_calls'] != 1:
        raise RuntimeError('Test evaluation count is not exactly one')
    if file_sha256(OUTPUT/'model.ubj') != validated['model_sha256']:
        raise RuntimeError('Model SHA mismatch')
    print({'status': result['status'], 'artifact_files': len(manifest['files']), 'test_evaluation_calls': 1})

In [ ]:
# CELL 14 — Display existing metrics; no model loading and no Test inference
if not OUTPUT.is_dir():
    print('No official result to display.')
else:
    m = result['metrics']; cm = m['confusion_matrix_labels_0_benign_1_attack']
    display({
        'experiment': result['experiment'], 'config': result['config'], 'backend': result['backend'],
        'best_k': result['best_k'], 'selected_features': result['selected_features'],
        'accuracy': m['accuracy'], 'macro_f1': m['macro_f1'],
        'balanced_accuracy': m['balanced_accuracy'],
        'recall_benign': m['recall_benign'], 'recall_attack': m['recall_attack_per_class'],
        'confusion_matrix_[Benign,Attack]': cm,
        'elapsed_seconds': result['elapsed_seconds'], 'peak_rss_bytes': result['peak_rss_bytes_sampled'],
        'xgboost_version': result['xgboost_version'], 'test_evaluation_calls': result['test_evaluation_calls'],
        'label_origin': result['label_origin'], 'capture_session_independence': result['capture_session_independence'],
    })